In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Setup + Download**

In [2]:
import os, json, gdown, zipfile
import numpy as np
import pandas as pd
import ast
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

os.makedirs("/kaggle/working/data",   exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/results",exist_ok=True)

# Download CSV
gdown.download(
    "https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye",
    "/kaggle/working/data/clean_multimodal_samples.csv",
    quiet=False)
print("✓ CSV downloaded")

✓ Device: Tesla T4


Downloading...
From: https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye
To: /kaggle/working/data/clean_multimodal_samples.csv
100%|██████████| 28.2M/28.2M [00:00<00:00, 71.9MB/s]

✓ CSV downloaded


# **Download + Extract Images**

In [3]:
print("Downloading ZebraMap images...")
gdown.download(
    "https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq",
    "/kaggle/working/zebramap_final.zip",
    quiet=False)

print("Extracting...")
os.makedirs("/kaggle/temp/images", exist_ok=True)
with zipfile.ZipFile(
        "/kaggle/working/zebramap_final.zip", 'r') as z:
    total = len(z.namelist())
    for i, member in enumerate(z.namelist()):
        z.extract(member, "/kaggle/temp/images")
        if i % 20000 == 0:
            print(f"  {i}/{total} ({i/total*100:.0f}%)")

os.remove("/kaggle/working/zebramap_final.zip")
total_imgs = sum(
    len(files) for _, _, files
    in os.walk("/kaggle/temp/images"))
print(f"✓ Total images: {total_imgs}")

Downloading...
From (original): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq
From (redirected): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq&confirm=t&uuid=4ca3c2a7-975c-4fd8-a470-1b55518db3a3
To: /kaggle/working/zebramap_final.zip
100%|██████████| 10.5G/10.5G [01:40<00:00, 104MB/s] 


Extracting...
  0/110262 (0%)
  20000/110262 (18%)
  40000/110262 (36%)
  60000/110262 (54%)
  80000/110262 (73%)
  100000/110262 (91%)
✓ Total images: 79478


# **Prepare Tier-A Data**

In [4]:
# Load dataset
df = pd.read_csv(
    "/kaggle/working/data/clean_multimodal_samples.csv")

new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Fix image paths for Kaggle
def fix_path(img_val):
    try:
        imgs = eval(img_val)
        for img in imgs:
            old = img['path']
            if 'images/' in old:
                rel = old.split('images/')[-1]
                img['path'] = \
                    f"/kaggle/temp/images/{rel}"
        return imgs
    except:
        return []

print("Fixing image paths...")
df['images_fixed'] = df['images'].apply(fix_path)

# ── Use Tier-A only for better accuracy ───────────────────────
# Download tiers from Drive
import gdown
gdown.download(
    "https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye",
    "/kaggle/working/data/clean_multimodal_samples.csv",
    quiet=True)

# Get unique orpha codes per disease
# Use top 88 most common diseases (Tier-A equivalent)
disease_counts = df['disease_name'].value_counts()
tier_a_diseases = disease_counts.head(88).index.tolist()
df_tier_a = df[
    df['disease_name'].isin(tier_a_diseases)
].reset_index(drop=True)

print(f"✓ Tier-A samples : {len(df_tier_a)}")
print(f"  Diseases       : {df_tier_a['disease_name'].nunique()}")

# Remap labels
label_counts = df_tier_a['label'].value_counts()
valid        = label_counts[label_counts >= 2].index
df_filtered  = df_tier_a[
    df_tier_a['label'].isin(valid)
].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered, test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old
                 in enumerate(all_labels)}
reverse_remap = {new: old for old, new
                 in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(
    subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

Fixing image paths...
✓ Tier-A samples : 17123
  Diseases       : 88
✓ Data ready
  Train   : 13698
  Test    : 3425
  Classes : 88


# **Dataset + DataLoader**

In [6]:
class ZebraFullDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                imgs = row['images_fixed']
                if not isinstance(imgs, list):
                    imgs = eval(imgs)
                for img_info in imgs:
                    if os.path.exists(img_info['path']):
                        self.samples.append({
                            'path' : img_info['path'],
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            img = Image.open(s['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(
                s['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3,
        saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225])
])

print("Building FULL datasets (no subsampling)...")
cnn_train_ds = ZebraFullDataset(
    train_df, train_transform)
cnn_test_ds  = ZebraFullDataset(
    test_df,  test_transform)

cnn_train_loader = DataLoader(
    cnn_train_ds, batch_size=64,
    shuffle=True,  num_workers=4)
cnn_test_loader  = DataLoader(
    cnn_test_ds,  batch_size=64,
    shuffle=False, num_workers=4)

print(f"✓ DataLoaders ready")
print(f"  Train : {len(cnn_train_ds)}")
print(f"  Test  : {len(cnn_test_ds)}")
print(f"  Batches: {len(cnn_train_loader)}")

Building FULL datasets (no subsampling)...
✓ DataLoaders ready
  Train : 11395
  Test  : 2821
  Batches: 179


# **Improved ResNet-50 + Class Weights**

In [9]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

# ── Rebuild sampler based on actual dataset samples ────────────
# cnn_train_ds.samples has the actual loaded images
# train_df has all rows but not all have valid images

# Get labels from actual loaded samples only
actual_labels = [s['label'] for s in cnn_train_ds.samples]
actual_labels_arr = np.array(actual_labels)

# Count per class
unique_classes, class_counts = np.unique(
    actual_labels_arr, return_counts=True)
class_weight = 1.0 / class_counts

# Assign weight to each sample
sample_weights = np.array([
    class_weight[np.where(unique_classes == lbl)[0][0]]
    for lbl in actual_labels_arr
])

sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# Rebuild loader with correct sampler
cnn_train_loader = DataLoader(
    cnn_train_ds, batch_size=64,
    sampler=sampler, num_workers=2  # reduced workers
)

print(f"✓ Weighted sampler fixed")
print(f"  Actual train samples : {len(cnn_train_ds.samples)}")
print(f"  Sampler size         : {len(sample_weights)}")
print(f"  Batches              : {len(cnn_train_loader)}")

# Also rebuild model + optimizer fresh
class ResNet50ClassifierV2(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(2048),
            nn.Dropout(dropout),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout * 0.75),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 20
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
cnn_model  = ResNet50ClassifierV2(NUM_CLASSES).to(device)
optimizer  = AdamW(
    filter(lambda p: p.requires_grad,
           cnn_model.parameters()),
    lr=5e-5, weight_decay=0.01)
scheduler  = CosineAnnealingLR(
    optimizer, T_max=CNN_EPOCHS, eta_min=1e-7)

print(f"✓ Model ready")
print(f"  Trainable params : "
      f"{sum(p.numel() for p in cnn_model.parameters() if p.requires_grad):,}")

✓ Weighted sampler fixed
  Actual train samples : 11395
  Sampler size         : 11395
  Batches              : 179
✓ Model ready
  Trainable params : 17,640,024


# **Train**

In [10]:
cnn_losses = []
cnn_accs   = []
best_acc   = 0
best_epoch = 0

print(f"Training CNN V2 Full Data — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels
                       ).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_losses.append(avg_loss)
    cnn_accs.append(acc)

    if acc > best_acc:
        best_acc   = acc
        best_epoch = epoch + 1
        torch.save(cnn_model.state_dict(),
                   '/kaggle/working/best_cnn_v2.pt')

    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%"
          + (" ← best" if epoch+1 == best_epoch else ""))

print("-" * 55)
print(f"✓ Training complete | Best: {best_acc:.2f}% "
      f"at epoch {best_epoch}")

# Load best
cnn_model.load_state_dict(
    torch.load('/kaggle/working/best_cnn_v2.pt'))

Training CNN V2 Full Data — 20 epochs
  Images  : 11395
  Classes : 88
-------------------------------------------------------
Epoch 01/20 | Loss: 4.4109 | Train Acc: 5.10% ← best
Epoch 02/20 | Loss: 4.0579 | Train Acc: 12.21% ← best
Epoch 03/20 | Loss: 3.8315 | Train Acc: 16.90% ← best
Epoch 04/20 | Loss: 3.6863 | Train Acc: 19.97% ← best
Epoch 05/20 | Loss: 3.5149 | Train Acc: 24.66% ← best
Epoch 06/20 | Loss: 3.3665 | Train Acc: 28.03% ← best
Epoch 07/20 | Loss: 3.2343 | Train Acc: 31.91% ← best
Epoch 08/20 | Loss: 3.1062 | Train Acc: 35.43% ← best
Epoch 09/20 | Loss: 3.0026 | Train Acc: 38.28% ← best
Epoch 10/20 | Loss: 2.8752 | Train Acc: 41.73% ← best
Epoch 11/20 | Loss: 2.7820 | Train Acc: 44.19% ← best
Epoch 12/20 | Loss: 2.7214 | Train Acc: 46.36% ← best
Epoch 13/20 | Loss: 2.6236 | Train Acc: 49.09% ← best
Epoch 14/20 | Loss: 2.5728 | Train Acc: 50.72% ← best
Epoch 15/20 | Loss: 2.5125 | Train Acc: 52.83% ← best
Epoch 16/20 | Loss: 2.4839 | Train Acc: 53.29% ← best
Epoch 17/2

<All keys matched successfully>

# **Evaluate + Save**

In [11]:
def evaluate_topk(model, loader, device, k=5):
    model.eval()
    all_preds, all_labels = [], []
    topk_correct = {1: 0, 3: 0, 5: 0}
    total = 0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            logits = model(images)

            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for k_val in [1, 3, 5]:
                k_act = min(k_val, logits.size(1))
                topk  = logits.topk(
                    k_act, dim=1).indices
                for i, lbl in enumerate(labels):
                    if lbl in topk[i]:
                        topk_correct[k_val] += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels,
                            all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "top1_accuracy": round(
            topk_correct[1]/total*100, 2),
        "top3_accuracy": round(
            topk_correct[3]/total*100, 2),
        "top5_accuracy": round(
            topk_correct[5]/total*100, 2),
        "total_samples": total
    }

print("Evaluating CNN V2...")
metrics = evaluate_topk(
    cnn_model, cnn_test_loader, device)

print("\n" + "=" * 55)
print("CNN V2 FULL DATA — RESULTS")
print("=" * 55)
print(f"  Accuracy     : {metrics['accuracy']}%"
      f"  (was 6.76%)")
print(f"  F1 Macro     : {metrics['f1_macro']}%")
print(f"  Top-1 Acc    : {metrics['top1_accuracy']}%")
print(f"  Top-3 Acc    : {metrics['top3_accuracy']}%")
print(f"  Top-5 Acc    : {metrics['top5_accuracy']}%")
print(f"  Test samples : {metrics['total_samples']}")
print(f"\n  Improvement  : "
      f"{metrics['accuracy']-6.76:+.2f}%")

# Save
import json

def convert(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict):
        return {convert(k): convert(v)
                for k,v in obj.items()}
    if isinstance(obj, list):
        return [convert(i) for i in obj]
    return obj

torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : convert(label_remap),
    'reverse_remap'   : convert(reverse_remap),
    'num_classes'     : int(NUM_CLASSES),
    'metrics'         : metrics,
    'architecture'    : 'ResNet50ClassifierV2'
}, "/kaggle/working/models/cnn_v2_full.pt")

with open("/kaggle/working/results/cnn_v2_summary.json",
          "w") as f:
    json.dump({
        "model"     : "ResNet50 V2 Full Data",
        "metrics"   : metrics,
        "baseline"  : {"accuracy": 6.76, "top5": 17.72},
        "gain"      : round(metrics['accuracy']-6.76, 2)
    }, f, indent=2)

print(f"\n✓ Model saved: cnn_v2_full.pt")
print(f"✓ Summary saved")
print(f"\nCNN V2 COMPLETE ✓")

Evaluating CNN V2...

CNN V2 FULL DATA — RESULTS
  Accuracy     : 14.89%  (was 6.76%)
  F1 Macro     : 13.52%
  Top-1 Acc    : 14.89%
  Top-3 Acc    : 30.24%
  Top-5 Acc    : 38.43%
  Test samples : 2821

  Improvement  : +8.13%

✓ Model saved: cnn_v2_full.pt
✓ Summary saved

CNN V2 COMPLETE ✓
